In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from netgen.geom2d import SplineGeometry
"""
der Code simuliert die Kirchhoff-love Gl mit HHJ-methode, nimmt eine 'verworfene' Platte, 
rechnete diese zusammen, speichert diese als *.xyz datei ab und plottet diese
"""

# l = 1100 #mm Länge
# b = 360 #mm Breite
# t  = 3.6     #Dicke [mm]
# step = 40    
# radius = 35 #mm Radius der Kreise für Auflager
       
def gravitationsEinfluss_Auflager(l,b,t,step,auflager_liste,radius):

    #region GEOMETRIE
    shape = MoveTo(0,0).Rectangle(l,b).Face()

    shape.edges.name="free"

    #circle = Circle((0,0),0.10).Face()
    #circle.edges.name="circles"

    rect = shape 

    #KREISAUFLAGER
    # auflager_liste = [(radius,radius),(l-radius,b-radius),(radius,b-radius),(l-radius,radius)]

    for mittelpunkt in auflager_liste:
        circle = Circle(mittelpunkt,radius).Face()
        circle.edges.name = "dirichlet"
        rect = rect - circle

    #KANTENAUFLAGER
    # rect.edges.Min(X).name = "dirichlet"
    # rect.edges.Max(X).name = "dirichlet"

    geo = OCCGeometry(rect,dim=2)

    #Draw(rect)
    mesh = Mesh(geo.GenerateMesh(maxh=l/40))

    mesh.Curve(3)
    #Draw(mesh)

    #endregion

    #region MATERIALKONSTANTEN
    E  = 70e6      #Glas ~ N'/mm² Elastizitätsmodul - ACHTUNG: N' nicht gleich N!!!      
    """
    UMRECHNUNG (m -> mm)

    Einheit Pascal: 1 Pa = 1 N/m² 
    Einheit Newton: 1 N = 1 kg*m/s²     - dh müssen auch Newton umrechnen

    Definiere:
    N' := kg*mm/s²      - in Millimeter!!!!
        -> 1 N = 1000 N'

    1 Pa = 1 N/m² = 10e-6 N/mm² = 10e-6 * 10e3 N'/mm² = 10e-3 N'/mm²

    Glas hat E-Modul von 70e9 Pa (70 GPa) = 70e9 N/m² = 70e9 * 10^(-3) N'/mm² = 70e6 N'/mm²         #neuer Wert
    """
    nu = 0.23       #dimensionslos, also bei m und mm gleich
    rho = 2.5e-6   #kg/mm³ Dichte 
    """
    Dichte von Glas 2500 kg/m³ = 2.5 * 10e3 kg/m³
    1kg/m³ = 1000g/10e9mm³ = 10e(-6)g/mm³
    -> 2.5 * 10e3 * 10e(-6) g/mm³ = 2.5 * 10e(-3) g/mm³ = 0.0025 g/mm³

    """

    g = 9810     # Erdbeschleunigung mm/s²
    """
    Erdbeschleunigung g = 9.81 m/s² = 9810 mm/s²
    """   

    q = rho * t * g     #Eigengewicht der Platte punktweise!!! - rechte Seite der PDE, also unser f_PDE

    #region FUNKTIONEN für PDE
    Db = E*t**3/(12*(1-nu**2))
    # Db = E*t**3/(12*(1+nu))             # In PHD paper anders

    # def D(A):
    #     return Db * (A + nu/(1-nu)*Trace(A)*Id(2))
    
    # def Dinv(A):
    #     return 1/Db * (A -nu*Trace(A)*Id(2)/(1+nu))


    def D(A):
        return Db *((1-nu)*A+ nu*Trace(A)*Id(2))

    def Dinv(A):
        return 1/Db * (1/(1-nu)*A - nu/(1-nu**2)*Trace(A)*Id(2))
    #endregion

    #region SIMULATION: FUNKTIONENRAUM,SCHWACHE FORM,VISUALISIERUNG
    order = 3

    V = HDivDiv(mesh, order=order-1,dirichlet="dirichlet")
    Q = H1(mesh, order=order, dirichlet="dirichlet")
    X = V * Q

    (sigma,w),(tau,v)= X.TnT()

    n = specialcf.normal(2)

    def tang(u):
        return u - (u*n)*n


    #schwache Form von Kirchhoff-Love mit Hellan-Herrmann-Johnson-Method
    #https://docu.ngsolve.org/ngs24/SaS/Kirchhoff_Love_plate.html

    a = BilinearForm(X,symmetric=True)
    a += InnerProduct(Dinv(sigma),tau) * dx
    a += div(sigma)*Grad(v) * dx
    a += div(tau) * Grad(w) * dx
    a += - (sigma[n,:] * tang(Grad(v)) + tau[n,:] * tang(Grad(w))  ) * dx(element_boundary=True)
    a.Assemble()

    L = LinearForm(X)
    L += q * v *dx
    L.Assemble()

    gf_solution = GridFunction(X)
    gf_solution.vec.data = a.mat.Inverse(X.FreeDofs(),inverse="") * L.vec

    gf_sigma, gf_w = gf_solution.components
    #Draw(gf_sigma, mesh,name="sigma")
    Draw(gf_w, mesh, name="disp",deformation=True, euler_angles=[-60,5,30])

    #endregion

    import numpy as np
    import inspect


    #erzeugt gitter für gewünschte werte
    # x_points = np.round(np.linspace(0,l, step)).astype(int)
    # y_points = np.round(np.linspace(0,b, step)).astype(int)
    x_points_ungerundet = np.linspace(0,l, step)
    x_points = np.round(x_points_ungerundet, 2)
    y_points_ungerundet = np.linspace(0,b, step)
    y_points = np.round(y_points_ungerundet, 2)
    # print(x_points_ungerundet)
    # print(x_points)

    X, Y = np.meshgrid(x_points, y_points)
    #print(x_points)

    #speichert (x,y,z) in gravitation_matrix für alle (x,y) in X,Y
    gravitation_matrix = np.zeros((step, step),dtype=object)
    for i in range(step):
        for j in range(step):
            try:
                z_eintrag = gf_w(X[i, j], Y[i, j])
            except:
                z_eintrag = 0
            gravitation_matrix[i, j] = (float(X[i,j]), float(Y[i,j]), z_eintrag)



    #*.xyz file erstellen
    filename_GRAV = f"{t}mm_gravitation.xyz"
    with open(filename_GRAV, "w") as f:
        for i in range(step):
            for j in range(step):
                x, y, w = gravitation_matrix[i, j]
                f.write(f"{x:.2f}\t{y:.2f}\t{w:.2f}\n")


    #Plotten

    # %matplotlib widget

    # import numpy as np
    # import matplotlib.pyplot as plt

    # np.set_printoptions(threshold=np.inf)

    # data_GRAV = np.loadtxt(filename_GRAV)

    # x_GRAV = data_GRAV[:,0]
    # y_GRAV = data_GRAV[:,1]
    # z_GRAV = data_GRAV[:,2]

    # # Raster erzeugen
    # X = np.unique(x_GRAV)
    # Y = np.unique(y_GRAV)

    # X, Y = np.meshgrid(X, Y)
    # Z_GRAV=z_GRAV.reshape(len(Y), len(X))

    # fig = plt.figure()
    # ax = fig.add_subplot(projection='3d')

    # ax.plot_surface(X, Y, Z_GRAV,color="blue")       # gravitation


    # ax.view_init(elev=10, azim=-75)
    # plt.show()

    return filename_GRAV